# Notebook 03: Census Redistribution to H3 Grid

Redistributes 2024 census population from manzana polygons to a uniform H3 hexagonal grid using area, sector, and road-network weights.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
import matplotlib.pyplot as plt
import h3
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

CRS_GEO = "EPSG:4674"
CRS_UTM = "EPSG:32719"
H3_RES  = 8

## Load census manzanas

In [ ]:
manzanas = gpd.read_file(CENSUS, layer="Manzanas_CPV24")
comunas  = gpd.read_file(CENSUS, layer="Comunal_CPV24")
r13_boundary = comunas.dissolve().geometry.iloc[0]

pop_total = manzanas["n_per"].sum()
print(f"Manzanas: {len(manzanas):,}")
print(f"Total population: {pop_total:,.0f}")
print(f"Manzanas with pop = 0: {(manzanas['n_per'] == 0).sum():,}")

## Load notebook 01 outputs

In [ ]:
h3_mapping = pd.read_parquet(DATA / "h3_sector_mapping.parquet")
bts = pd.read_parquet(DATA / "bts_sites.parquet")

print(f"H3 sector mapping: {len(h3_mapping):,} rows, "
      f"{h3_mapping['h3_index'].nunique():,} unique H3 cells")

## Polyfill R13 with H3 grid

In [ ]:
def polyfill_polygon(geom, resolution):
    cells = set()
    polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
    for poly in polys:
        outer = [(lat, lng) for lng, lat in poly.exterior.coords]
        holes = [[(lat, lng) for lng, lat in hole.coords] for hole in poly.interiors]
        h3_poly = h3.LatLngPoly(outer, *holes)
        cells |= set(h3.polygon_to_cells(h3_poly, resolution))
    return cells

h3_cells = polyfill_polygon(r13_boundary, H3_RES)
print(f"H3 res-{H3_RES} cells covering R13: {len(h3_cells):,}")

h3_polys = []
for idx in h3_cells:
    boundary = h3.cell_to_boundary(idx)
    poly = Polygon([(lng, lat) for lat, lng in boundary])
    h3_polys.append({"h3_index": idx, "geometry": poly})

h3_gdf = gpd.GeoDataFrame(h3_polys, crs=CRS_GEO)
h3_gdf["h3_area_km2"] = h3_gdf.to_crs(CRS_UTM).geometry.area / 1e6

print(f"H3 grid built: {len(h3_gdf):,} hexagons")

## Area-weighted redistribution (baseline)

In [ ]:
mz = manzanas[manzanas["n_per"] > 0][["MANZENT", "n_per", "geometry"]].copy()
mz = mz.to_crs(CRS_UTM)
h3_utm = h3_gdf.to_crs(CRS_UTM)

print(f"Overlaying {len(mz):,} populated manzanas with {len(h3_utm):,} H3 cells...")
overlay = gpd.overlay(mz, h3_utm, how="intersection")
overlay["intersection_area"] = overlay.geometry.area

mz_total_area = overlay.groupby("MANZENT")["intersection_area"].transform("sum")
overlay["weight_area"] = overlay["intersection_area"] / mz_total_area
overlay["pop_area"] = overlay["n_per"] * overlay["weight_area"]

h3_pop_area = overlay.groupby("h3_index")["pop_area"].sum().reset_index()

print(f"H3 cells with population: {len(h3_pop_area):,}")
print(f"Total redistributed: {h3_pop_area['pop_area'].sum():,.0f} (census: {pop_total:,.0f})")

## Sector coverage score per H3 cell

In [ ]:
nb01_h3_centers = []
for _, row in h3_mapping.iterrows():
    lat, lng = h3.cell_to_latlng(row["h3_index"])
    nb01_h3_centers.append({
        "bts_id": row["bts_id"],
        "azimuth": row["azimuth"],
        "lat": lat,
        "lng": lng,
    })
nb01_centers = pd.DataFrame(nb01_h3_centers)

nb01_centers["h3_res8"] = [
    h3.latlng_to_cell(row["lat"], row["lng"], H3_RES)
    for _, row in nb01_centers.iterrows()
]

sector_coverage = (
    nb01_centers.groupby("h3_res8")[["bts_id", "azimuth"]]
    .apply(lambda g: len(g.drop_duplicates()))
    .reset_index(name="n_covering_sectors")
    .rename(columns={"h3_res8": "h3_index"})
)

print(f"Res-8 hexes with sector coverage: {len(sector_coverage):,}")

## Sector-weighted redistribution

In [ ]:
overlay_sw = overlay.merge(sector_coverage, on="h3_index", how="left")
overlay_sw["n_covering_sectors"] = overlay_sw["n_covering_sectors"].fillna(0)

overlay_sw["weight_sector"] = (
    overlay_sw["intersection_area"] * (1 + overlay_sw["n_covering_sectors"])
)

mz_total_sw = overlay_sw.groupby("MANZENT")["weight_sector"].transform("sum")
overlay_sw["weight_sector_norm"] = overlay_sw["weight_sector"] / mz_total_sw
overlay_sw["pop_sector"] = overlay_sw["n_per"] * overlay_sw["weight_sector_norm"]

h3_pop_sector = overlay_sw.groupby("h3_index")["pop_sector"].sum().reset_index()

print(f"Sector-weighted total: {h3_pop_sector['pop_sector'].sum():,.0f}")

## Road-network refined redistribution

In [ ]:
import osmnx as ox

roads_file = DATA / "osm_roads_r13.gpkg"

try:
    if roads_file.exists():
        print(f"Loading cached roads from {roads_file}")
        roads = gpd.read_file(roads_file)
    else:
        print("Downloading OSM road network for R13...")
        G = ox.graph_from_polygon(r13_boundary, network_type="drive")
        roads = ox.graph_to_gdfs(G, nodes=False)
        roads.to_file(roads_file, driver="GPKG")

    roads_utm = roads.to_crs(CRS_UTM)
    roads["length_m"] = roads_utm.geometry.length
    centroids = roads.geometry.centroid
    roads["h3_index"] = [
        h3.latlng_to_cell(p.y, p.x, H3_RES) for p in centroids
    ]

    road_density = (
        roads.groupby("h3_index")["length_m"]
        .sum()
        .reset_index()
        .rename(columns={"length_m": "road_length_m"})
    )
    HAS_ROADS = True

except Exception as e:
    print(f"Road download failed: {e}")
    HAS_ROADS = False

In [ ]:
h3_pop_gdf = h3_gdf.merge(h3_pop_area, on="h3_index", how="left")
h3_pop_gdf["pop_area"] = h3_pop_gdf["pop_area"].fillna(0)
h3_pop_gdf = h3_pop_gdf.merge(h3_pop_sector, on="h3_index", how="left")
h3_pop_gdf["pop_sector"] = h3_pop_gdf["pop_sector"].fillna(0)

if HAS_ROADS:
    overlay_rd = overlay_sw.merge(road_density, on="h3_index", how="left")
    overlay_rd["road_length_m"] = overlay_rd["road_length_m"].fillna(0)

    overlay_rd["weight_road"] = (
        overlay_rd["intersection_area"]
        * (1 + overlay_rd["n_covering_sectors"])
        * (1 + overlay_rd["road_length_m"] / 1000)
    )

    mz_total_rd = overlay_rd.groupby("MANZENT")["weight_road"].transform("sum")
    overlay_rd["weight_road_norm"] = overlay_rd["weight_road"] / mz_total_rd
    overlay_rd["pop_road"] = overlay_rd["n_per"] * overlay_rd["weight_road_norm"]

    h3_pop_road = overlay_rd.groupby("h3_index")["pop_road"].sum().reset_index()

    h3_pop_gdf = h3_pop_gdf.merge(h3_pop_road, on="h3_index", how="left")
    h3_pop_gdf["pop_road"] = h3_pop_gdf["pop_road"].fillna(0)
else:
    h3_pop_gdf["pop_road"] = h3_pop_gdf["pop_sector"]

# Validate totals
print("=== Population totals ===")
print(f"Census:           {pop_total:,.0f}")
print(f"Area-weighted:    {h3_pop_gdf['pop_area'].sum():,.0f}")
print(f"Sector-weighted:  {h3_pop_gdf['pop_sector'].sum():,.0f}")
print(f"Road-refined:     {h3_pop_gdf['pop_road'].sum():,.0f}")

## Save outputs

In [ ]:
h3_save = h3_pop_gdf[["h3_index", "h3_area_km2",
                       "pop_area", "pop_sector", "pop_road"]].copy()

lats, lngs = [], []
for h in h3_save["h3_index"]:
    lat, lng = h3.cell_to_latlng(h)
    lats.append(lat)
    lngs.append(lng)
h3_save["lat"] = lats
h3_save["lon"] = lngs

h3_save.to_parquet(DATA / "h3_population.parquet", index=False)
print(f"Saved: h3_population.parquet ({len(h3_save):,} rows)")

sector_coverage.to_parquet(DATA / "h3_sector_coverage.parquet", index=False)
print(f"Saved: h3_sector_coverage.parquet ({len(sector_coverage):,} rows)")